# Experiment Orchestrator - Classifier Selection

Orquesta el experimento de selección de clasificador para el pipeline.

**6 configuraciones** = 3 backbones (ResNet-18, EfficientNet-B0, DenseNet-121) × 2 métodos:
- **euclidean**: prototipos Euclidianos, backbone **congelado** (sin entrenar).
- **meta**: prototipos con **meta-training episódico** del backbone (EasyFSL).

Few-shot por prototipos (EasyFSL `PrototypicalNetworks`, 2 prototipos: glaucoma/normal). Support set **balanced** (N glaucoma + N normal), barrido **N ∈ {1,2,3,4,5}**, **5 seeds** compartidas.

**Estrategia de seeds:** en la iteración `i`, TODAS las configs usan `seeds[i]` → mismas imágenes, comparación justa. Resultados = `mean ± std` sobre las 5 seeds.

**Score de selección (SPEC §8):** `0.40 × F1-macro (val) + 0.40 × IoU Grad-CAM + 0.20 × (1 − VRAM_norm)`. La mejor de las 6 configs va al pipeline.

### Celda 0: Import repo and setup Drive

In [ ]:
# --- Bootstrap Colab: código + datos + working dir ---
import os
REPO = "/content/Medgemma_Segmentation_CIARP_2026"
if not os.path.isdir(REPO):
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 -b Classifier-Implementation https://github.com/TheBug95/Medgemma_Segmentation_CIARP_2026.git {REPO}
%cd {REPO}
!GIT_LFS_SKIP_SMUDGE=1 git pull origin Classifier-Implementation      # traer la última versión (código + notebook)
!pip install -q easyfsl                                    # few-shot por prototipos (EasyFSL)
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/REFUGE_data.zip -d Datasets/REFUGE/   # datos reales sobre los punteros
%cd Experiments/Classifier_Selection/notebook
import torch; print("GPU:", torch.cuda.is_available())

## Celda 1: Setup

Instala dependencias e importa módulos.

In [ ]:
# =====================================================================
# Celda 1: Setup — imports, config y helper de evaluación Grad-CAM
# =====================================================================
# Colab ya trae torch/torchvision/PIL/yaml/numpy/scipy. easyfsl se instala en la
# celda de bootstrap.

import sys
import os
import json

import numpy as np
import yaml
import torch

# Se asume que el notebook corre desde .../Classifier_Selection/notebook/
sys.path.insert(0, '..')

from modules.data_module import DataModule
from modules.cnn_classifier import CNNClassifier
from modules.prototype_classifier import PrototypeClassifier
from scripts.few_shot import train_few_shot
from scripts.benchmark_inference import run_benchmark

# --- Configuración -----------------------------------------------------------
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

SEEDS     = config['few_shot']['seeds']       # [42, 123, 456, 789, 1024]
N_SAMPLES = config['few_shot']['n_samples']   # [1, 2, 3, 4, 5]
BACKBONES = config['backbones']               # ['resnet18', 'efficientnet_b0', 'densenet121']
METHODS   = config['few_shot']['methods']     # ['euclidean', 'meta']
N_KEYS    = [f'N{n}' for n in N_SAMPLES]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Seeds: {SEEDS}')
print(f'N_samples: {N_SAMPLES}  | support_mode: {config["few_shot"].get("support_mode", "balanced")}')
print(f'Backbones: {BACKBONES}  | Métodos: {METHODS}  → {len(BACKBONES)*len(METHODS)} configs')


# =====================================================================
# Helper: cargar el clasificador guardado para evaluar su Grad-CAM
# =====================================================================
def load_classifier_for_gradcam(backbone, method, path, config, seed):
    """Reconstruye y carga el clasificador (según método) para extraer Grad-CAM."""
    nc = config['classifier']['num_classes']
    if method == 'finetune':
        clf = CNNClassifier({'backbone': backbone, 'num_classes': nc,
                             'pretrained': False, 'seed': seed})
    else:  # euclidean | meta -> prototipos. pretrained=True: para 'euclidean' el
           # backbone es ImageNet; para 'meta' load() lo sobreescribe con los pesos guardados.
        clf = PrototypeClassifier({'backbone': backbone, 'num_classes': nc,
                                   'pretrained': True, 'seed': seed})
    clf.load(path)
    return clf


# =====================================================================
# Helper: CALIDAD del Grad-CAM (IoU + pointing) vs máscara GT
# =====================================================================
# Se usa el get_gradcam PROPIO de cada clasificador (patrón forward-hook + tensor
# hook), que funciona con los 3 backbones. NO se usa modules/gradcam_module.py
# (compartido) porque su register_full_backward_hook choca con el relu in-place de
# densenet en el torch de Colab. El algoritmo es el MISMO para euclidean, meta y
# finetune, así que la comparación de Grad-CAM sigue siendo consistente.

def _iou(pred_binary, gt_binary):
    """IoU entre dos máscaras binarias (H, W)."""
    pred = pred_binary.astype(bool)
    gt = gt_binary.astype(bool)
    union = np.logical_or(pred, gt).sum()
    return float(np.logical_and(pred, gt).sum() / union) if union else 0.0


def _pointing_hit(heatmap, gt_binary):
    """1.0 si el píxel más caliente del heatmap cae dentro de la GT, 0.0 si no."""
    y, x = np.unravel_index(int(np.argmax(heatmap)), heatmap.shape)
    return 1.0 if gt_binary[y, x] > 0 else 0.0


def evaluate_gradcam_quality(model, data_loader, config, device):
    """
    Mide la calidad del Grad-CAM (de la clase 'glaucoma') vs la máscara GT del
    disco óptico sobre todo el data_loader. Binariza por percentil para el IoU.

    Returns: {mean_iou, mean_pointing_accuracy, iou_per_sample, pointing_per_sample}
    """
    percentile = config['gradcam'].get('percentile', 95)
    glaucoma_idx = (model.class_names.index('glaucoma')
                    if 'glaucoma' in model.class_names else 1)

    ious, pointings = [], []
    for batch in data_loader:
        images = batch['image']
        masks = batch['mask']
        for i in range(images.size(0)):
            gt = masks[i, 0].numpy()                               # (H, W) binaria {0,1}
            heatmap = model.get_gradcam(images[i], target_class=glaucoma_idx)
            threshold = np.percentile(heatmap, percentile)
            binary = (heatmap >= threshold).astype(np.float32)
            ious.append(_iou(binary, gt))
            pointings.append(_pointing_hit(heatmap, gt))

    return {
        'mean_iou': float(np.mean(ious)) if ious else 0.0,
        'mean_pointing_accuracy': float(np.mean(pointings)) if pointings else 0.0,
        'iou_per_sample': ious,
        'pointing_per_sample': pointings,
    }

## Celda 2: Convert Data

Convierte el dataset REFUGE al formato del proyecto.

In [ ]:
# Convertir REFUGE → annotations.json + splits.json
import subprocess
subprocess.run(['python', '../scripts/convert_refuge_format.py'], check=True)

## Celda 3: Initialize DataModule

Carga los datos. El DataModule maneja la lógica de qué imágenes son glaucoma.

In [ ]:
# Inicializar DataModule. Pasamos seed y augmentations explícitos para que
# vengan del config.yaml (y no de los defaults internos del módulo).
data_cfg = {**config['data'], 'seed': config['seed'], 'augmentations': config['augmentations']}
data_module = DataModule(data_cfg)
val_loader  = data_module.get_val_loader()

# Verificar que el split de train tiene exactamente 40 glaucoma
train_glaucoma = data_module.get_glaucoma_indices(split='train')
print(f'Val: {len(val_loader.dataset)} muestras (glaucoma + normal)')
print(f'Glaucoma disponibles en train: {len(train_glaucoma)} (máximo N factible = {len(train_glaucoma)-1})')
assert len(train_glaucoma) >= max(N_SAMPLES), \
    f'ERROR: Solo hay {len(train_glaucoma)} glaucoma pero se pide N={max(N_SAMPLES)}'

## Celda 4: Run Experiment — 6 configs × 5 seeds

**Bucles:** seed → backbone → método (`euclidean`, `meta`).

Para cada config y cada N: se ajusta un clasificador fresco (euclidean = solo prototipos; meta = meta-training episódico del backbone + prototipos), se evalúa **F1-macro en el val completo** (40 glaucoma + 360 normal), y se mide la calidad del **Grad-CAM** (IoU + pointing vs máscara GT del disco óptico) con el modelo del N más grande.

**Nota:** el modo `meta` entrena el backbone (`n_episodes` episodios por cada N), así que tarda más que `euclidean`.

In [ ]:
# Estructura: results[backbone][method][seed] = {few_shot: {N1..N5}, gradcam}
results = {b: {m: {} for m in METHODS} for b in BACKBONES}
largest_n_key = N_KEYS[-1]   # el N más grande (p.ej. 'N5'), usado para el Grad-CAM

for seed in SEEDS:
    print(f'\n{"="*65}')
    print(f'ITERACIÓN seed={seed}  |  Todas las configs usarán esta seed')
    print(f'{"="*65}')

    for backbone in BACKBONES:
        for method in METHODS:
            print(f'\n  ── {backbone} | {method} | seed={seed}')

            # [1/2] Few-shot (todos los N): ajusta/entrena, evalúa en val, guarda.
            #   euclidean -> prototipos congelados | meta -> meta-training + prototipos
            print(f'    [1/2] {method}: N={N_SAMPLES}...')
            few_shot_result = train_few_shot(backbone, data_module, config, seed=seed, method=method)

            # [2/2] Calidad del Grad-CAM con el modelo del N más grande (opción Y:
            #   classifier.get_gradcam, patrón forward-hook seguro para densenet).
            print(f'    [2/2] Grad-CAM ({largest_n_key}) vs máscara GT...')
            model_path = f'../results/{backbone}/{method}/seed_{seed}/model_{largest_n_key}.pth'
            model = load_classifier_for_gradcam(backbone, method, model_path, config, seed)
            gradcam_metrics = evaluate_gradcam_quality(model, val_loader, config, device)

            results[backbone][method][seed] = {
                'few_shot': few_shot_result,
                'gradcam':  gradcam_metrics,
            }
            print(f'    ✓ {backbone}/{method} seed={seed} | '
                  f'IoU={gradcam_metrics["mean_iou"]:.3f} '
                  f'pointing={gradcam_metrics["mean_pointing_accuracy"]:.3f}')

# Benchmark computacional por ARQUITECTURA (independiente de método/seed/N:
# mide el costo del backbone, que es el mismo para frozen y meta).
print(f'\n{"="*65}')
print('BENCHMARK COMPUTACIONAL (una vez por backbone)')
print(f'{"="*65}')
computational = {}
for backbone in BACKBONES:
    model = CNNClassifier({'backbone': backbone,
                           'num_classes': config['classifier']['num_classes'],
                           'pretrained': False,
                           'seed': 42})
    computational[backbone] = run_benchmark(
        backbone, model, val_loader, device,
        num_runs=config['benchmark']['num_runs'],
    )
    print(f'  ✓ {backbone}: {computational[backbone]["total_parameters"]/1e6:.1f}M params, '
          f'{computational[backbone]["vram_batch_1_mb"]:.0f} MB VRAM')

## Celda 5: Agregar Resultados y Determinar Winner

Calcula `mean ± std` sobre las 5 seeds para cada una de las **6 configs** (backbone × método) y su score de selección.

**Fórmula (SPEC §8):**
```
Score = 0.40 × mean(F1@N1..N5) + 0.40 × IoU_GradCAM + 0.20 × (1 - VRAM_norm)
```
La VRAM es por arquitectura (igual para frozen y meta) y se normaliza entre las 6 configs. **Winner** = la config con mayor score → es la que va al pipeline.

In [ ]:
import numpy as np

# Una "config" = (backbone, method). 6 en total (3 backbones × 2 métodos).
CONFIGS = [(b, m) for b in BACKBONES for m in METHODS]

summary = {}
for backbone, method in CONFIGS:
    cfg_key = f'{backbone}/{method}'
    seed_results = results[backbone][method]   # {seed: {few_shot, gradcam}}

    # Métricas por seed para cada N
    f1_by_n = {nk: [seed_results[s]['few_shot'][nk]['f1_macro'] for s in SEEDS]
               for nk in N_KEYS}
    iou_list = [seed_results[s]['gradcam']['mean_iou'] for s in SEEDS]

    per_seed = {}
    for s in SEEDS:
        per_seed[str(s)] = {'mean_iou_gradcam': seed_results[s]['gradcam']['mean_iou']}
        for nk in N_KEYS:
            per_seed[str(s)][f'f1_{nk}'] = seed_results[s]['few_shot'][nk]['f1_macro']

    aggregated = {
        'mean_iou_mean': float(np.mean(iou_list)),
        'mean_iou_std':  float(np.std(iou_list)),
    }
    for nk in N_KEYS:
        aggregated[f'f1_{nk}_mean'] = float(np.mean(f1_by_n[nk]))
        aggregated[f'f1_{nk}_std']  = float(np.std(f1_by_n[nk]))

    summary[cfg_key] = {
        'backbone':      backbone,
        'method':        method,
        'per_seed':      per_seed,
        'aggregated':    aggregated,
        'computational': computational[backbone],
    }

# Score por config (VRAM normalizada entre las 6 configs)
all_vrams = [summary[f'{b}/{m}']['computational']['vram_batch_1_mb'] for b, m in CONFIGS]
for backbone, method in CONFIGS:
    cfg_key = f'{backbone}/{method}'
    agg  = summary[cfg_key]['aggregated']
    vram = summary[cfg_key]['computational']['vram_batch_1_mb']

    f1_mean   = float(np.mean([agg[f'f1_{nk}_mean'] for nk in N_KEYS]))   # media sobre N1..N5
    iou_mean  = agg['mean_iou_mean']
    vram_norm = (vram - min(all_vrams)) / (max(all_vrams) - min(all_vrams) + 1e-8)

    summary[cfg_key]['score'] = float(0.40 * f1_mean + 0.40 * iou_mean + 0.20 * (1 - vram_norm))

winner = max(summary, key=lambda k: summary[k]['score'])

# Tabla de resultados (una fila por config)
print(f'\nRESULTADOS FINALES (mean ± std sobre {len(SEEDS)} seeds):')
header = f'{"Config":<28}' + ''.join(f'  {nk:>11}' for nk in N_KEYS) + f'  {"IoU":>11}  {"Score":>8}'
print(header)
print('-' * len(header))
for cfg_key in summary:
    a = summary[cfg_key]['aggregated']
    row = f'{cfg_key:<28}'
    for nk in N_KEYS:
        row += f"  {a[f'f1_{nk}_mean']:.3f}±{a[f'f1_{nk}_std']:.3f}"
    row += f"  {a['mean_iou_mean']:.3f}±{a['mean_iou_std']:.3f}"
    row += f"  {summary[cfg_key]['score']:.4f}"
    print(row)
print(f'\n🏆 Winner: {winner} (score={summary[winner]["score"]:.4f})')

# Guardar selection_summary.json
import datetime
final = {
    'configs':     summary,
    'winner':      winner,
    'seeds_used':  SEEDS,
    'n_samples':   N_SAMPLES,
    'methods':     METHODS,
    'config_used': config,
    'timestamp':   datetime.datetime.utcnow().isoformat() + 'Z',
}
os.makedirs('../results', exist_ok=True)
with open('../results/selection_summary.json', 'w') as f:
    json.dump(final, f, indent=2)
print('\n✓ Guardado: results/selection_summary.json')

### Save results on Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, datetime
src = '/content/Medgemma_Segmentation_CIARP_2026/Experiments/Classifier_Selection/results'
dst = '/content/drive/MyDrive/CIARP_results_' + datetime.datetime.now().strftime('%Y%m%d_%H%M')
shutil.copytree(src, dst)
print('✅ Copiado a Drive:', dst)